# Sistemas de Recomendação

In [29]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

dim_treino= pd.read_csv("../Dados Finais/dim_treino.csv", encoding="utf-8")
dim_aula= pd.read_csv("../Dados Finais/dim_aula.csv", encoding="utf-8")
dim_cliente= pd.read_csv("../Dados Finais/dim_cliente.csv", encoding="utf-8")
fact_cliente= pd.read_csv("../Dados Finais/tf_cliente.csv", encoding="utf-8")

## Recomendações de Treinos (Content Based)

In [30]:
# 1. Codificar dificuldade, objetivo e incluir duração/calorias
treinos_encoded = pd.get_dummies(
    dim_treino[['treino_sk', 'treino_calorias', 'treino_duracao_min', 'treino_dificuldade', 'treino_objetivo']],
    columns=['treino_dificuldade', 'treino_objetivo']
)

# 2. Perfil médio do cliente (média dos atributos dos treinos que já fez)
cliente_treinos = fact_cliente.merge(treinos_encoded, left_on='sk_treino', right_on='treino_sk')
perfil_cliente_treino = cliente_treinos.groupby('sk_cliente').mean(numeric_only=True)[treinos_encoded.columns[1:]]  # Exclui treino_sk

# 3. Perfil dos treinos (exclui treino_sk)
perfil_treino = treinos_encoded.set_index('treino_sk')[treinos_encoded.columns[1:]]

# 4. Identificar colunas quantitativas e qualitativas
quantitativas = ['treino_calorias', 'treino_duracao_min']
objetivo_cols = [col for col in perfil_cliente_treino.columns if 'treino_objetivo' in col]
dificuldade_cols = [col for col in perfil_cliente_treino.columns if 'treino_dificuldade' in col]

# 5. Normalizar apenas quantitativas
scaler = StandardScaler()
perfil_cliente_treino_quant = pd.DataFrame(
    scaler.fit_transform(perfil_cliente_treino[quantitativas]),
    index=perfil_cliente_treino.index,
    columns=quantitativas
)
perfil_treino_quant = pd.DataFrame(
    scaler.transform(perfil_treino[quantitativas]),
    index=perfil_treino.index,
    columns=quantitativas
)

# 6. Multiplicar qualitativas pelo peso desejado
peso_objetivo = 7
peso_dificuldade = 3

perfil_cliente_treino_obj = perfil_cliente_treino[objetivo_cols] * peso_objetivo
perfil_treino_obj = perfil_treino[objetivo_cols] * peso_objetivo

perfil_cliente_treino_dif = perfil_cliente_treino[dificuldade_cols] * peso_dificuldade
perfil_treino_dif = perfil_treino[dificuldade_cols] * peso_dificuldade

# 7. Concatenar quantitativas normalizadas + qualitativas ponderadas
perfil_cliente_final = pd.concat([perfil_cliente_treino_quant, perfil_cliente_treino_obj, perfil_cliente_treino_dif], axis=1)
perfil_treino_final = pd.concat([perfil_treino_quant, perfil_treino_obj, perfil_treino_dif], axis=1)

# 8. Similaridade e recomendações
sim_matrix_treino = cosine_similarity(perfil_cliente_final, perfil_treino_final)
sim_df_treino = pd.DataFrame(sim_matrix_treino, index=perfil_cliente_final.index, columns=perfil_treino_final.index)

# 9. Recomendar treinos para um cliente exemplo
cliente_id = perfil_cliente_final.index[0]  # ou outro cliente
treinos_feitos = fact_cliente[fact_cliente['sk_cliente'] == cliente_id]['sk_treino'].dropna().unique()
recomendacoes_treino = sim_df_treino.loc[cliente_id].drop(labels=treinos_feitos, errors='ignore').sort_values(ascending=False)
print(f"Top recomendações de treinos para o cliente {cliente_id}:")
print(recomendacoes_treino.head())

# 10. Explicação da principal recomendação
treino_recomendado = recomendacoes_treino.index[0]
perfil_treino_recomendado = perfil_treino.loc[treino_recomendado]
perfil_cliente = perfil_cliente_treino.loc[cliente_id]

# Objetivo mais frequente do cliente e do treino recomendado
objetivos_cliente = perfil_cliente[objetivo_cols].sort_values(ascending=False)
objetivo_principal_cliente = objetivos_cliente.index[0].replace('treino_objetivo_', '')
objetivos_treino = perfil_treino_recomendado[objetivo_cols]
objetivo_principal_treino = objetivos_treino[objetivos_treino == 1].index[0].replace('treino_objetivo_', '')

# Dificuldade mais frequente do cliente e do treino recomendado
dificuldades_cliente = perfil_cliente[dificuldade_cols].sort_values(ascending=False)
dificuldade_principal_cliente = dificuldades_cliente.index[0].replace('treino_dificuldade_', '')
dificuldades_treino = perfil_treino_recomendado[dificuldade_cols]
dificuldade_principal_treino = dificuldades_treino[dificuldades_treino == 1].index[0].replace('treino_dificuldade_', '')

# Calorias e duração
calorias_cliente = perfil_cliente['treino_calorias']
calorias_treino = perfil_treino_recomendado['treino_calorias']
duracao_cliente = perfil_cliente['treino_duracao_min']
duracao_treino = perfil_treino_recomendado['treino_duracao_min']

# Obter mapeamento treino_sk -> treino_nome
sk_to_nome = dim_treino.set_index('treino_sk')['treino_nome'].to_dict()

# Mostrar recomendações com nome do treino
top_n = 3
print(f"Top {top_n} recomendações de treinos para o cliente {cliente_id}:")
for treino_sk, score in recomendacoes_treino.head(top_n).items():
    print(f"- {sk_to_nome.get(treino_sk, treino_sk)} (score: {score:.3f})")

# Explicação para os 3 principais treinos recomendados
for treino_sk in recomendacoes_treino.head(top_n).index:
    perfil_treino_recomendado = perfil_treino.loc[treino_sk]
    # Objetivo
    objetivos_treino = perfil_treino_recomendado[objetivo_cols]
    objetivo_principal_treino = objetivos_treino[objetivos_treino == 1].index[0].replace('treino_objetivo_', '')
    # Dificuldade
    dificuldades_treino = perfil_treino_recomendado[dificuldade_cols]
    dificuldade_principal_treino = dificuldades_treino[dificuldades_treino == 1].index[0].replace('treino_dificuldade_', '')
    # Calorias e duração
    calorias_treino = perfil_treino_recomendado['treino_calorias']
    duracao_treino = perfil_treino_recomendado['treino_duracao_min']
    print(f"""
Explicação para o treino '{sk_to_nome.get(treino_sk, treino_sk)}':
- O cliente pratica muitos treinos com objetivo '{objetivo_principal_cliente}' e dificuldade '{dificuldade_principal_cliente}'.
- O treino recomendado tem objetivo '{objetivo_principal_treino}' e dificuldade '{dificuldade_principal_treino}'.
- O cliente costuma queimar em média {calorias_cliente:.0f} calorias e treinos de {duracao_cliente:.0f} minutos.
- O treino recomendado queima {calorias_treino:.0f} calorias e tem duração de {duracao_treino:.0f} minutos.
""")

Top recomendações de treinos para o cliente 1:
treino_sk
14    0.633504
6     0.373846
7     0.370830
8     0.361740
9     0.352755
Name: 1, dtype: float64
Top 3 recomendações de treinos para o cliente 1:
- Circuito de Força (score: 0.634)
- Circuito de Abdómen (score: 0.374)
- Circuito de Cardio (score: 0.371)

Explicação para o treino 'Circuito de Força':
- O cliente pratica muitos treinos com objetivo 'Perda de peso' e dificuldade 'Alto'.
- O treino recomendado tem objetivo 'Perda de peso' e dificuldade 'Alto'.
- O cliente costuma queimar em média 531 calorias e treinos de 39 minutos.
- O treino recomendado queima 641 calorias e tem duração de 36 minutos.


Explicação para o treino 'Circuito de Abdómen':
- O cliente pratica muitos treinos com objetivo 'Perda de peso' e dificuldade 'Alto'.
- O treino recomendado tem objetivo 'Definição muscular' e dificuldade 'Médio'.
- O cliente costuma queimar em média 531 calorias e treinos de 39 minutos.
- O treino recomendado queima 355 calorias

## Recomendações de Aulas (Content Based)

In [31]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# 1. Codificar intensidade e incluir duração/calorias
aulas_encoded = pd.get_dummies(
    dim_aula[['aula_sk', 'aula_calorias', 'aula_duracao_min', 'aula_intensidade']],
    columns=['aula_intensidade']
)

# 2. Perfil médio do cliente (média dos atributos das aulas que já fez)
cliente_aulas = fact_cliente.merge(aulas_encoded, left_on='sk_aula', right_on='aula_sk')
perfil_cliente_aula = cliente_aulas.groupby('sk_cliente').mean(numeric_only=True)[aulas_encoded.columns[1:]]  # Exclui aula_sk

# 3. Perfil das aulas (exclui aula_sk)
perfil_aula = aulas_encoded.set_index('aula_sk')[aulas_encoded.columns[1:]]

# 4. Identificar colunas quantitativas e qualitativas
quantitativas_aula = ['aula_calorias', 'aula_duracao_min']
intensidade_cols = [col for col in perfil_cliente_aula.columns if 'aula_intensidade' in col]

# 5. Normalizar apenas quantitativas
scaler_aula = StandardScaler()
perfil_cliente_aula_quant = pd.DataFrame(
    scaler_aula.fit_transform(perfil_cliente_aula[quantitativas_aula]),
    index=perfil_cliente_aula.index,
    columns=quantitativas_aula
)
perfil_aula_quant = pd.DataFrame(
    scaler_aula.transform(perfil_aula[quantitativas_aula]),
    index=perfil_aula.index,
    columns=quantitativas_aula
)

# 6. Multiplicar qualitativas pelo peso desejado
peso_intensidade = 5
perfil_cliente_aula_int = perfil_cliente_aula[intensidade_cols] * peso_intensidade
perfil_aula_int = perfil_aula[intensidade_cols] * peso_intensidade

# 7. Concatenar quantitativas normalizadas + qualitativas ponderadas
perfil_cliente_aula_final = pd.concat([perfil_cliente_aula_quant, perfil_cliente_aula_int], axis=1)
perfil_aula_final = pd.concat([perfil_aula_quant, perfil_aula_int], axis=1)

# 8. Similaridade e recomendações
sim_matrix_aula = cosine_similarity(perfil_cliente_aula_final, perfil_aula_final)
sim_df_aula = pd.DataFrame(sim_matrix_aula, index=perfil_cliente_aula_final.index, columns=perfil_aula_final.index)

# 9. Recomendar aulas para um cliente exemplo
cliente_id = perfil_cliente_aula_final.index[0]  # ou outro cliente
aulas_feitas = fact_cliente[fact_cliente['sk_cliente'] == cliente_id]['sk_aula'].dropna().unique()
recomendacoes_aula = sim_df_aula.loc[cliente_id].drop(labels=aulas_feitas, errors='ignore').sort_values(ascending=False)

# Obter mapeamento aula_sk -> aula_nome
sk_to_nome_aula = dim_aula.set_index('aula_sk')['aula_nome'].to_dict()

# Mostrar recomendações com nome da aula
top_n = 3
print(f"Top {top_n} recomendações de aulas para o cliente {cliente_id}:")
for aula_sk, score in recomendacoes_aula.head(top_n).items():
    print(f"- {sk_to_nome_aula.get(aula_sk, aula_sk)} (score: {score:.3f})")

# Explicação para os 3 principais aulas recomendadas
if cliente_id not in perfil_cliente_aula.index:
    print(f"O cliente {cliente_id} ainda não praticou aulas. Não é possível gerar recomendações personalizadas.")
else:
    for aula_sk in recomendacoes_aula.head(top_n).index:
        perfil_aula_recomendada = perfil_aula.loc[aula_sk]
        # Intensidade da aula recomendada
        intensidades_aula = perfil_aula_recomendada[intensidade_cols]
        intensidade_principal_aula = intensidades_aula[intensidades_aula == 1]
        intensidade_principal_aula = intensidade_principal_aula.index[0].replace('aula_intensidade_', '') if not intensidades_aula[intensidades_aula == 1].empty else 'Desconhecido'
        # Intensidade mais frequente do cliente
        intensidades_cliente = perfil_cliente_aula.loc[cliente_id, intensidade_cols]
        intensidades_cliente_sorted = intensidades_cliente.sort_values(ascending=False)
        intensidade_principal_cliente = intensidades_cliente_sorted.index[0].replace('aula_intensidade_', '') if not intensidades_cliente_sorted.empty else 'Desconhecido'
        # Calorias e duração
        calorias_cliente = perfil_cliente_aula['aula_calorias'].loc[cliente_id]
        calorias_aula = perfil_aula_recomendada['aula_calorias']
        duracao_cliente = perfil_cliente_aula['aula_duracao_min'].loc[cliente_id]
        duracao_aula = perfil_aula_recomendada['aula_duracao_min']
        print(f"""
    Explicação para a aula '{sk_to_nome_aula.get(aula_sk, aula_sk)}':
    - O cliente pratica muitas aulas com intensidade '{intensidade_principal_cliente}'.
    - A aula recomendada tem intensidade '{intensidade_principal_aula}'.
    - O cliente costuma queimar em média {calorias_cliente:.0f} calorias e aulas de {duracao_cliente:.0f} minutos.
    - A aula recomendada queima {calorias_aula:.0f} calorias e tem duração de {duracao_aula:.0f} minutos.
    """)

Top 3 recomendações de aulas para o cliente 1:
- Aula de Pilates (score: 0.936)
- Aula de Funcional (score: 0.664)
- Aula de Yoga (score: 0.280)

    Explicação para a aula 'Aula de Pilates':
    - O cliente pratica muitas aulas com intensidade 'Média'.
    - A aula recomendada tem intensidade 'Média'.
    - O cliente costuma queimar em média 386 calorias e aulas de 48 minutos.
    - A aula recomendada queima 369 calorias e tem duração de 56 minutos.
    

    Explicação para a aula 'Aula de Funcional':
    - O cliente pratica muitas aulas com intensidade 'Média'.
    - A aula recomendada tem intensidade 'Média'.
    - O cliente costuma queimar em média 386 calorias e aulas de 48 minutos.
    - A aula recomendada queima 543 calorias e tem duração de 31 minutos.
    

    Explicação para a aula 'Aula de Yoga':
    - O cliente pratica muitas aulas com intensidade 'Média'.
    - A aula recomendada tem intensidade 'Baixa'.
    - O cliente costuma queimar em média 386 calorias e aulas de 48

## Recomendações de Treinos (Collaborative and Demographic hybrid)

In [32]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Perfil de objetivos/dificuldades por cliente
cliente_treinos = fact_cliente.merge(dim_treino, left_on='sk_treino', right_on='treino_sk')
perfil_objetivo = pd.get_dummies(cliente_treinos['treino_objetivo'])
perfil_dificuldade = pd.get_dummies(cliente_treinos['treino_dificuldade'])
cliente_treinos = pd.concat([cliente_treinos, perfil_objetivo, perfil_dificuldade], axis=1)
perfil_cliente_obj = cliente_treinos.groupby('sk_cliente')[perfil_objetivo.columns].mean()
perfil_cliente_dif = cliente_treinos.groupby('sk_cliente')[perfil_dificuldade.columns].mean()

# 2. Demográficos: faixa etária, IMC, profissão
demograficos = dim_cliente.set_index('cliente_sk')[['cliente_faixa_etaria', 'cliente_profissao', 'cliente_peso_kg', 'cliente_altura_cm']]
demograficos['imc'] = demograficos['cliente_peso_kg'] / ((demograficos['cliente_altura_cm'] / 100) ** 2)

# One-hot encoding para faixa etária
ohe_faixa = OneHotEncoder(sparse=False)
faixa_encoded = ohe_faixa.fit_transform(demograficos[['cliente_faixa_etaria']])
faixa_encoded_df = pd.DataFrame(faixa_encoded, index=demograficos.index, columns=ohe_faixa.get_feature_names_out(['cliente_faixa_etaria']))

# One-hot encoding para profissão
ohe_prof = OneHotEncoder(sparse=False)
prof_encoded = ohe_prof.fit_transform(demograficos[['cliente_profissao']])
prof_encoded_df = pd.DataFrame(prof_encoded, index=demograficos.index, columns=ohe_prof.get_feature_names_out(['cliente_profissao']))

# Concatenar tudo
demograficos_final = pd.concat([faixa_encoded_df, prof_encoded_df], axis=1)
demograficos_final['imc'] = StandardScaler().fit_transform(demograficos[['imc']])

# 3. Concatenar tudo para vetor de perfil do cliente
perfil_cliente = pd.concat([perfil_cliente_obj, perfil_cliente_dif, demograficos_final], axis=1).fillna(0)

# 4. Similaridade user-user
sim_matrix = cosine_similarity(perfil_cliente)
sim_df = pd.DataFrame(sim_matrix, index=perfil_cliente.index, columns=perfil_cliente.index)

# 5. Recomendar treinos feitos por clientes mais semelhantes
cliente_id = perfil_cliente.index[0]
clientes_semelhantes = sim_df.loc[cliente_id].sort_values(ascending=False).index[1:6]
treinos_feitos = set(fact_cliente[fact_cliente['sk_cliente'] == cliente_id]['sk_treino'].dropna())
treinos_vizinhos = set(fact_cliente[fact_cliente['sk_cliente'].isin(clientes_semelhantes)]['sk_treino'].dropna())
recomendaveis = treinos_vizinhos - treinos_feitos

# (Opcional) Ordenar por frequência entre vizinhos
scores = {t: (fact_cliente[(fact_cliente['sk_cliente'].isin(clientes_semelhantes)) & (fact_cliente['sk_treino'] == t)].shape[0]) for t in recomendaveis}
top_recomendacoes = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:3]

# Mostrar nomes dos treinos recomendados
sk_to_nome = dim_treino.set_index('treino_sk')['treino_nome'].to_dict()
print(f"Top recomendações de treinos para o cliente {cliente_id}:")
for treino_sk, score in top_recomendacoes:
    print(f"- {sk_to_nome.get(treino_sk, treino_sk)} (nº de vizinhos que fizeram: {score})")

# Explicação das recomendações
for treino_sk, score in top_recomendacoes:
    treino_nome = sk_to_nome.get(treino_sk, treino_sk)
    # Objetivo e dificuldade do treino recomendado
    treino_row = dim_treino[dim_treino['treino_sk'] == treino_sk].iloc[0]
    objetivo_treino = treino_row['treino_objetivo']
    dificuldade_treino = treino_row['treino_dificuldade']
    # Perfil do cliente
    cliente_row = dim_cliente[dim_cliente['cliente_sk'] == cliente_id].iloc[0]
    faixa_cliente = cliente_row['cliente_faixa_etaria']
    prof_cliente = cliente_row['cliente_profissao']
    imc_cliente = demograficos.loc[cliente_id, 'imc']
    # Objetivo e dificuldade mais frequentes do cliente
    obj_cliente = perfil_cliente_obj.loc[cliente_id].idxmax()
    dif_cliente = perfil_cliente_dif.loc[cliente_id].idxmax()
    print(f"""
Explicação para o treino '{treino_nome}':
- O cliente pertence à faixa etária '{faixa_cliente}', profissão '{prof_cliente}', IMC {imc_cliente:.1f}.
- O cliente pratica muitos treinos com objetivo '{obj_cliente}' e dificuldade '{dif_cliente}'.
- O treino recomendado tem objetivo '{objetivo_treino}' e dificuldade '{dificuldade_treino}'.
- {score} dos clientes mais semelhantes também fizeram este treino.
""")

Top recomendações de treinos para o cliente 1:
- Circuito de Cardio (nº de vizinhos que fizeram: 5)
- Circuito de Ombros (nº de vizinhos que fizeram: 4)
- Circuito de Corpo inteiro (nº de vizinhos que fizeram: 3)

Explicação para o treino 'Circuito de Cardio':
- O cliente pertence à faixa etária '30-44', profissão 'Estudante', IMC 16.4.
- O cliente pratica muitos treinos com objetivo 'Perda de peso' e dificuldade 'Alto'.
- O treino recomendado tem objetivo 'Resistência' e dificuldade 'Médio'.
- 5 dos clientes mais semelhantes também fizeram este treino.


Explicação para o treino 'Circuito de Ombros':
- O cliente pertence à faixa etária '30-44', profissão 'Estudante', IMC 16.4.
- O cliente pratica muitos treinos com objetivo 'Perda de peso' e dificuldade 'Alto'.
- O treino recomendado tem objetivo 'Condicionamento físico' e dificuldade 'Médio'.
- 4 dos clientes mais semelhantes também fizeram este treino.


Explicação para o treino 'Circuito de Corpo inteiro':
- O cliente pertence à fa

c:\Users\afons\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\preprocessing\_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
c:\Users\afons\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\preprocessing\_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


## Recomendações de Aulas (Collaborative and Demographic hybrid)

In [33]:
# 1. Perfil de intensidade por cliente
cliente_aulas = fact_cliente.merge(dim_aula, left_on='sk_aula', right_on='aula_sk')
perfil_intensidade = pd.get_dummies(cliente_aulas['aula_intensidade'])
cliente_aulas = pd.concat([cliente_aulas, perfil_intensidade], axis=1)
perfil_cliente_int = cliente_aulas.groupby('sk_cliente')[perfil_intensidade.columns].mean()

# 2. Demográficos: faixa etária, IMC, profissão
demograficos = dim_cliente.set_index('cliente_sk')[['cliente_faixa_etaria', 'cliente_profissao', 'cliente_peso_kg', 'cliente_altura_cm']]
demograficos['imc'] = demograficos['cliente_peso_kg'] / ((demograficos['cliente_altura_cm'] / 100) ** 2)

# One-hot encoding para faixa etária
ohe_faixa = OneHotEncoder(sparse=False)
faixa_encoded = ohe_faixa.fit_transform(demograficos[['cliente_faixa_etaria']])
faixa_encoded_df = pd.DataFrame(faixa_encoded, index=demograficos.index, columns=ohe_faixa.get_feature_names_out(['cliente_faixa_etaria']))

# One-hot encoding para profissão
ohe_prof = OneHotEncoder(sparse=False)
prof_encoded = ohe_prof.fit_transform(demograficos[['cliente_profissao']])
prof_encoded_df = pd.DataFrame(prof_encoded, index=demograficos.index, columns=ohe_prof.get_feature_names_out(['cliente_profissao']))

# Concatenar tudo
demograficos_final = pd.concat([faixa_encoded_df, prof_encoded_df], axis=1)
demograficos_final['imc'] = StandardScaler().fit_transform(demograficos[['imc']])

# 3. Concatenar tudo para vetor de perfil do cliente
perfil_cliente = pd.concat([perfil_cliente_int, demograficos_final], axis=1).fillna(0)

# 4. Similaridade user-user
sim_matrix = cosine_similarity(perfil_cliente)
sim_df = pd.DataFrame(sim_matrix, index=perfil_cliente.index, columns=perfil_cliente.index)

# 5. Recomendar aulas feitas por clientes mais semelhantes
cliente_id = perfil_cliente.index[0]
clientes_semelhantes = sim_df.loc[cliente_id].sort_values(ascending=False).index[1:6]
aulas_feitas = set(fact_cliente[fact_cliente['sk_cliente'] == cliente_id]['sk_aula'].dropna())
aulas_vizinhos = set(fact_cliente[fact_cliente['sk_cliente'].isin(clientes_semelhantes)]['sk_aula'].dropna())
recomendaveis = aulas_vizinhos - aulas_feitas

# (Opcional) Ordenar por frequência entre vizinhos
scores = {a: (fact_cliente[(fact_cliente['sk_cliente'].isin(clientes_semelhantes)) & (fact_cliente['sk_aula'] == a)].shape[0]) for a in recomendaveis}
top_recomendacoes = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:3]

# Mostrar nomes das aulas recomendadas
sk_to_nome_aula = dim_aula.set_index('aula_sk')['aula_nome'].to_dict()
print(f"Top recomendações de aulas para o cliente {cliente_id}:")
for aula_sk, score in top_recomendacoes:
    print(f"- {sk_to_nome_aula.get(aula_sk, aula_sk)} (nº de vizinhos que fizeram: {score})")

# Explicação das recomendações
for aula_sk, score in top_recomendacoes:
    aula_nome = sk_to_nome_aula.get(aula_sk, aula_sk)
    # Intensidade da aula recomendada
    aula_row = dim_aula[dim_aula['aula_sk'] == aula_sk].iloc[0]
    intensidade_aula = aula_row['aula_intensidade']
    # Perfil do cliente
    cliente_row = dim_cliente[dim_cliente['cliente_sk'] == cliente_id].iloc[0]
    faixa_cliente = cliente_row['cliente_faixa_etaria']
    prof_cliente = cliente_row['cliente_profissao']
    imc_cliente = demograficos.loc[cliente_id, 'imc']
    # Intensidade mais frequente do cliente
    int_cliente = perfil_cliente_int.loc[cliente_id].idxmax()
    print(f"""
Explicação para a aula '{aula_nome}':
- O cliente pertence à faixa etária '{faixa_cliente}', profissão '{prof_cliente}', IMC {imc_cliente:.1f}.
- O cliente pratica muitas aulas com intensidade '{int_cliente}'.
- A aula recomendada tem intensidade '{intensidade_aula}'.
- {score} dos clientes mais semelhantes também fizeram esta aula.
""")

Top recomendações de aulas para o cliente 1:
- Aula de Funcional (nº de vizinhos que fizeram: 8)
- Aula de Alongamentos (nº de vizinhos que fizeram: 6)
- Aula de Musculação (nº de vizinhos que fizeram: 5)

Explicação para a aula 'Aula de Funcional':
- O cliente pertence à faixa etária '30-44', profissão 'Estudante', IMC 16.4.
- O cliente pratica muitas aulas com intensidade 'Média'.
- A aula recomendada tem intensidade 'Média'.
- 8 dos clientes mais semelhantes também fizeram esta aula.


Explicação para a aula 'Aula de Alongamentos':
- O cliente pertence à faixa etária '30-44', profissão 'Estudante', IMC 16.4.
- O cliente pratica muitas aulas com intensidade 'Média'.
- A aula recomendada tem intensidade 'Alta'.
- 6 dos clientes mais semelhantes também fizeram esta aula.


Explicação para a aula 'Aula de Musculação':
- O cliente pertence à faixa etária '30-44', profissão 'Estudante', IMC 16.4.
- O cliente pratica muitas aulas com intensidade 'Média'.
- A aula recomendada tem intensidad

c:\Users\afons\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\preprocessing\_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
c:\Users\afons\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\preprocessing\_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
